In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# from datasets import load_from_disk
# from PIL import Image
# from pathlib import Path
# from tqdm import tqdm
# import hashlib

# def extract_samples(dataset_path):
#     ds = load_from_disk(str(dataset_path))
#     samples = {}
#     for row in tqdm(ds, desc=f"Extracting from {Path(dataset_path).name}"):
#         image = row.get("image")
#         caption = row.get("text")
#         if isinstance(image, Image.Image) and caption:
#             hashname = hashlib.md5(image.tobytes()).hexdigest()
#             img_filename = f"{hashname}.jpg"
#             samples[img_filename] = caption
#     return samples

In [ ]:
# import json
# from pathlib import Path

# # Paths
# ar_jsonl_path = "/content/drive/MyDrive/RecoMind/Processed Data/ar_caption_dataset.jsonl"
# output_jsonl = "/content/drive/MyDrive/RecoMind/Processed Data/merged_caption_dataset.jsonl"

# # English datasets
# en_dataset_paths = [
#     "/content/drive/MyDrive/RecoMind/Raw Data/clothes_desc",
#     "/content/drive/MyDrive/RecoMind/Raw Data/h-and-m-fashion-caption"
# ]

# english_captions = {}
# for path in en_dataset_paths:
#     english_captions.update(extract_samples(path))

# with open(ar_jsonl_path, 'r', encoding='utf-8') as infile, \
#      open(output_jsonl, 'w', encoding='utf-8') as outfile:

#     for line in infile:
#         entry = json.loads(line)
#         img_filename = Path(entry["image"]).name
#         entry["text_en"] = english_captions.get(img_filename, "")
#         outfile.write(json.dumps(entry, ensure_ascii=False) + '\n')

# print("Done: English captions merged successfully.")

Extracting from h-and-m-fashion-caption: 100%|██████████| 20491/20491 [10:49<00:00, 31.53it/s]


Done: English captions merged successfully.


In [15]:
import json
import re
from pathlib import Path

input_jsonl_path = Path("/content/drive/MyDrive/RecoMind/Processed Data/merged_caption_dataset.jsonl")
output_jsonl_path = Path("/content/drive/MyDrive/RecoMind/Processed Data/dataset_cleaned.jsonl")

def has_repeated_words(text, repeat_threshold=3):
    pattern = re.compile(r'(\b\w+\b)(\s+\1){' + str(repeat_threshold-1) + r',}', re.UNICODE)
    return bool(pattern.search(text))

def fix_arabic_text(text):
    # Remove "التطبيق / 3d"
    text = re.sub(r'التطبيق\s*/\s*3d', '', text, flags=re.IGNORECASE)
    # Remove English words
    text = re.sub(r'\b[a-zA-Z]+\b', '', text)
    # Replace size x size
    text = re.sub(r'(\d+)\s*x\s*(\d+)', r'\1 في \2', text)
    # Replace "V"
    text = re.sub(r'\bV\b', 'شكل مثلث', text)
    # Remove escaped quotes \"
    text = text.replace('\\"', '')
    # Replace "قمة"
    text = text.replace('قمة', 'قميص')
    # Clean extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def fix_english_text(text_en):
    # Remove "application/3d"
    text_en = re.sub(r'application\s*/\s*3d', '', text_en, flags=re.IGNORECASE)
    text_en = re.sub(r'\s+', ' ', text_en).strip()
    return text_en

with input_jsonl_path.open('r', encoding='utf-8') as infile, \
     output_jsonl_path.open('w', encoding='utf-8') as outfile:

    removed_count = 0
    total_count = 0

    for line in infile:
        total_count += 1
        entry = json.loads(line)

        if has_repeated_words(entry['text'], repeat_threshold=3):
            removed_count += 1
            continue

        entry['text'] = fix_arabic_text(entry['text'])

        if 'text_en' in entry:
            entry['text_en'] = fix_english_text(entry['text_en'])

        outfile.write(json.dumps(entry, ensure_ascii=False) + '\n')

print(f"Total entries processed: {total_count}")
print(f"Entries removed due to repeated words: {removed_count}")
print(f"Cleaned dataset saved to: {output_jsonl_path}")


Total entries processed: 21491
Entries removed due to repeated words: 334
Cleaned dataset saved to: /content/drive/MyDrive/RecoMind/Processed Data/dataset_cleaned.jsonl


In [16]:
import json

jsonl_path = "/content/drive/MyDrive/RecoMind/Processed Data/dataset_cleaned.jsonl"

with open(jsonl_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        entry = json.loads(line)
        print(f"Entry {i+1}:")
        print("Image:", entry.get("image"))
        print("Text:", entry.get("text"))
        print("Text_en:", entry.get("text_en", "No English caption"))
        print("-" * 40)
        if i >= 11:
            break

Entry 1:
Image: /content/drive/MyDrive/RecoMind/Raw Data/clothes_desc/ac1a7dc14fad31d1795f2d19e9717ed9.jpg
Text: الكافتان الوردي طول العجل المنسوج في خليط مع عنق وأزرار مخفية أسفل الجبهة. نير مزدوج الطبقات يواصل أسفل الأكمام، وملفوفات في الجبهة وحزام ربطة عنق في الخصر. غير مقيد.
Text_en: Pink Calf-length kaftan woven in a Tencel™ lyocell blend with a V-neck and concealed buttons down the front. Double-layered yoke that continues down the sleeves, pleats at the front and a tie belt at the waist. Unlined.
----------------------------------------
Entry 2:
Image: /content/drive/MyDrive/RecoMind/Raw Data/clothes_desc/937bef632cb4f3f5f97fbbed9a3d61c7.jpg
Text: قميص برتقال برتقال خفيف في كشمير ناعم مع طوق يقف، الأكمام طويلة راجلان مع الأصفاد المربطة، والحافة المستقيمة.
Text_en: Light Orange Rib-knit jumper in soft cashmere with a stand-up collar, long raglan sleeves with ribbed cuffs, and a straight hem.
----------------------------------------
Entry 3:
Image: /content/drive/MyDrive/RecoMind/